# Entrenamiento YOLO26 — Dataset `final_ucsma_train` (Tesis UCSM)

**Equipo:** RTX 5060 Laptop 8 GB · i7-14650HX · 32 GB RAM · Windows
**Carpeta de trabajo:** `TRAIN_FINAL_UCSM_YOLO26`

Flujo del cuaderno:
1. Configuración de rutas e hiperparámetros
2. Instalación (PyTorch **CUDA 12.8**, obligatorio para RTX serie 50) — solo la primera vez
3. Verificación de la GPU
4. Descarga del dataset desde Roboflow
5. Corrección de `data.yaml` y análisis del dataset
6. Entrenamiento (con opción de reanudar)
7. Evaluación en validación y prueba (métricas para la tesis)
8. Velocidad de inferencia (GPU FP16 y CPU)
9. *(Opcional)* Comparación YOLO26n vs YOLO26s para elegir el más eficiente
10. Exportación (ONNX / TensorRT) y copia del modelo final al proyecto Django
11. Prueba visual y gráficas

> Guarda este `.ipynb` **dentro** de `TRAIN_FINAL_UCSM_YOLO26` y ejecuta las celdas en orden.

## 1. Configuración
Edita solo esta celda. Los valores por defecto están pensados para **8 GB de VRAM** y para obtener el modelo más eficiente (YOLO26n: el más rápido, ideal para tiempo real con la ESP32-CAM).

In [ ]:
from pathlib import Path
import os

# ---------------- Rutas ----------------
BASE_DIR = Path(r"C:\Users\Usuario\PycharmProjects\TRAIN_YOLO26\TRAIN_FINAL_UCSM_YOLO26")
if not BASE_DIR.exists():                      # si el cuaderno se abrió desde otra ruta
    BASE_DIR = Path.cwd()
DATASET_DIR    = BASE_DIR / "dataset"          # aquí se descarga el dataset
RUNS_DIR       = BASE_DIR / "runs"             # resultados de Ultralytics
RESULTADOS_DIR = BASE_DIR / "resultados_tesis" # métricas en CSV/JSON para la tesis
MODELO_FINAL_DIR = BASE_DIR / "modelo_final"   # best.pt, ONNX y engine finales
DESTINO_DJANGO = Path(r"D:\Tesis_2026\SV_INTERFAZ_UCSM\modelos_ia")  # se copia si existe

# ---------------- Roboflow ----------------
RF_WORKSPACE = "darwin-turpo-s-workspace"
RF_PROYECTO  = "final_ucsma_train"
RF_VERSION   = 1
RF_FORMATO   = "yolo26"

# ---------------- Modelo ----------------
# yolo26n = el más eficiente (velocidad/precisión). yolo26s = más preciso, ~2-3x más lento.
MODELO_BASE = "yolo26n.pt"
PESOS_LOCALES = BASE_DIR.parent / MODELO_BASE  # usa el yolo26n.pt que ya tienes, si existe

# ---------------- Entrenamiento ----------------
NOMBRE_EXP = "ucsm_yolo26n"
IMGSZ      = 640      # igual a la resolución VGA de la ESP32-CAM
EPOCHS     = 200      # con early stopping (PATIENCE) se detiene antes si ya no mejora
PATIENCE   = 40
BATCH      = 0.75     # fracción de VRAM (autobatch al 75 %). Si da "out of memory", usa 16
WORKERS    = 8        # si en Windows aparece "DataLoader worker exited", pon 0 o 2
SEMILLA    = 42

# ---------------- Opciones ----------------
INSTALAR_DEPENDENCIAS = False   # True solo la primera vez (luego reinicia el kernel)
ENTRENAR_COMPARACION  = False   # True para entrenar también yolo26s y comparar
EXPORTAR_TENSORRT     = True    # intenta exportar .engine FP16 (máxima velocidad en GPU)

for d in (RUNS_DIR, RESULTADOS_DIR, MODELO_FINAL_DIR):   # DATASET_DIR lo crea Roboflow
    d.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", BASE_DIR)

## 2. Instalación (solo la primera vez)
La RTX 5060 (arquitectura **Blackwell, sm_120**) necesita **PyTorch compilado con CUDA 12.8** (`cu128`) y driver NVIDIA **≥ 570**. Con el PyTorch normal de `pip` la GPU no funciona.

Pon `INSTALAR_DEPENDENCIAS = True`, ejecuta la celda y luego **reinicia el kernel** (Kernel → Restart). Después vuelve a poner `False`.

In [ ]:
import sys, subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

if INSTALAR_DEPENDENCIAS:
    pip("install", "--upgrade", "pip")
    pip("uninstall", "-y", "torch", "torchvision", "torchaudio")
    pip("install", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu128")
    pip("install", "-U", "ultralytics", "roboflow", "pandas", "matplotlib", "pyyaml", "onnx", "onnxslim")
    print("\n>>> Instalación terminada. REINICIA EL KERNEL y pon INSTALAR_DEPENDENCIAS = False")
else:
    print("Instalación omitida (INSTALAR_DEPENDENCIAS = False)")

## 3. Verificación de la GPU

In [ ]:
import torch, platform

print("Python     :", platform.python_version())
print("PyTorch    :", torch.__version__, "| CUDA del build:", torch.version.cuda)
print("CUDA disp. :", torch.cuda.is_available())
assert torch.cuda.is_available(), "PyTorch no ve la GPU: ejecuta la celda de instalación (cu128) y reinicia el kernel."

nombre = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU        : {nombre} | compute capability {cap[0]}.{cap[1]} | VRAM {vram:.1f} GB")
print("Arquitecturas soportadas por este PyTorch:", torch.cuda.get_arch_list())
if cap >= (12, 0) and not any(a.startswith("sm_12") for a in torch.cuda.get_arch_list()):
    raise RuntimeError("Este PyTorch no soporta Blackwell (sm_120). Instala la versión cu128.")

# Prueba rápida de cómputo en GPU (FP16)
a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
torch.cuda.synchronize(); b = a @ a; torch.cuda.synchronize()
print("Prueba de multiplicación de matrices en GPU: OK")
del a, b; torch.cuda.empty_cache()

import ultralytics
ultralytics.checks()

## 4. Descarga del dataset (Roboflow)
La API key **no** se escribe en el cuaderno: se lee de la variable de entorno `ROBOFLOW_API_KEY` o se pide al ejecutar.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

# Si la carpeta existe pero no tiene data.yaml (descarga omitida o incompleta), se fuerza la descarga
ya_descargado = DATASET_DIR.exists() and any(DATASET_DIR.rglob("data.yaml"))
if not ya_descargado:
    api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("API key de Roboflow: ")
    rf = Roboflow(api_key=api_key)
    proyecto = rf.workspace(RF_WORKSPACE).project(RF_PROYECTO)
    version = proyecto.version(RF_VERSION)
    dataset = version.download(RF_FORMATO, location=str(DATASET_DIR), overwrite=True)
    print("Dataset descargado en:", dataset.location)
else:
    print("El dataset ya estaba descargado.")

# Si Roboflow lo dejó en una subcarpeta, se ubica automáticamente
DATASET_DIR = next(DATASET_DIR.rglob("data.yaml")).parent
print("Usando DATASET_DIR =", DATASET_DIR)
print("Contenido:", [p.name for p in DATASET_DIR.iterdir()])

## 5. Corrección de `data.yaml` y análisis del dataset
Roboflow suele escribir rutas relativas (`../train/images`) que fallan en Windows. Se genera `data_ucsm.yaml` con rutas absolutas. Si el dataset no trae carpeta de validación, se separa un 15 % del entrenamiento.

In [ ]:
import yaml, random, shutil
from collections import Counter

EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def carpeta_split(*nombres):
    for n in nombres:
        p = DATASET_DIR / n / "images"
        if p.exists():
            return p
    return None

with open(DATASET_DIR / "data.yaml", encoding="utf-8") as f:
    cfg_rf = yaml.safe_load(f)
NOMBRES = cfg_rf["names"]
if isinstance(NOMBRES, dict):
    NOMBRES = [NOMBRES[k] for k in sorted(NOMBRES)]

train_dir = carpeta_split("train")
val_dir   = carpeta_split("valid", "val")
test_dir  = carpeta_split("test")
assert train_dir is not None, "No se encontró train/images"

if val_dir is None:   # crear validación con el 15 % de train
    print("No hay validación: se separa el 15 % de train")
    val_dir = DATASET_DIR / "valid" / "images"
    (DATASET_DIR / "valid" / "labels").mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(p for p in train_dir.iterdir() if p.suffix.lower() in EXT_IMG)
    random.Random(SEMILLA).shuffle(imgs)
    for img in imgs[: int(len(imgs) * 0.15)]:
        lbl = train_dir.parent / "labels" / (img.stem + ".txt")
        shutil.move(str(img), val_dir / img.name)
        if lbl.exists():
            shutil.move(str(lbl), DATASET_DIR / "valid" / "labels" / lbl.name)

datos = {
    "path": str(DATASET_DIR),
    "train": str(train_dir.relative_to(DATASET_DIR)),
    "val": str(val_dir.relative_to(DATASET_DIR)),
    "nc": len(NOMBRES),
    "names": NOMBRES,
}
if test_dir is not None:
    datos["test"] = str(test_dir.relative_to(DATASET_DIR))

DATA_YAML = DATASET_DIR / "data_ucsm.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(datos, f, allow_unicode=True, sort_keys=False)
TIENE_TEST = test_dir is not None
print(open(DATA_YAML, encoding="utf-8").read())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def analizar_split(img_dir):
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in EXT_IMG]
    lbl_dir = img_dir.parent / "labels"
    conteo, vacias, sin_label = Counter(), 0, 0
    for img in imgs:
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            sin_label += 1
            continue
        lineas = [l for l in lbl.read_text().splitlines() if l.strip()]
        if not lineas:
            vacias += 1
        for l in lineas:
            conteo[int(l.split()[0])] += 1
    return len(imgs), conteo, vacias, sin_label

filas, conteo_total = [], Counter()
for nombre, d in [("train", train_dir), ("val", val_dir), ("test", test_dir)]:
    if d is None:
        continue
    n, c, vac, sin = analizar_split(d)
    conteo_total += c
    filas.append({"split": nombre, "imagenes": n, "instancias": sum(c.values()),
                  "imagenes_sin_objetos": vac, "imagenes_sin_label": sin})
resumen_dataset = pd.DataFrame(filas)
display(resumen_dataset)

clases = pd.DataFrame({"clase": NOMBRES,
                       "instancias": [conteo_total.get(i, 0) for i in range(len(NOMBRES))]})
display(clases)
resumen_dataset.to_csv(RESULTADOS_DIR / "dataset_resumen.csv", index=False)
clases.to_csv(RESULTADOS_DIR / "dataset_clases.csv", index=False)

ax = clases.plot.bar(x="clase", y="instancias", legend=False, color="#0cb967", figsize=(8, 4))
ax.set_title("Instancias por clase (todo el dataset)"); ax.set_ylabel("instancias")
plt.tight_layout(); plt.savefig(RESULTADOS_DIR / "dataset_clases.png", dpi=150); plt.show()

minimo = clases["instancias"].min()
if minimo < 0.1 * clases["instancias"].max():
    print("AVISO: hay clases muy desbalanceadas; considera más imágenes de las clases minoritarias.")

In [ ]:
# Cache: en RAM si cabe (más rápido), si no en disco
n_total = int(resumen_dataset["imagenes"].sum())
gb_estimados = n_total * IMGSZ * IMGSZ * 3 / 1024**3
CACHE = "ram" if gb_estimados < 10 else "disk"
print(f"{n_total} imágenes ≈ {gb_estimados:.1f} GB en caché → CACHE = '{CACHE}'")

## 6. Entrenamiento
Configuración orientada a eficiencia: AMP (FP16 mixto), *autobatch* al 75 % de la VRAM, *cosine LR*, *early stopping* y cierre del mosaico en las últimas 15 épocas. El mejor modelo se guarda como `best.pt`.

In [ ]:
from ultralytics import YOLO
import time, json

def entrenar(modelo_base, nombre_exp):
    pesos = str(PESOS_LOCALES) if (modelo_base == MODELO_BASE and PESOS_LOCALES.exists()) else modelo_base
    modelo = YOLO(pesos)
    t0 = time.time()
    modelo.train(
        data=str(DATA_YAML),
        epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
        device=0, workers=WORKERS, cache=CACHE, amp=True,
        optimizer="auto", cos_lr=True, close_mosaic=15,
        seed=SEMILLA, deterministic=False,
        project=str(RUNS_DIR), name=nombre_exp, exist_ok=False,
        plots=True, save=True, val=True, verbose=True,
    )
    horas = (time.time() - t0) / 3600
    save_dir = Path(modelo.trainer.save_dir)
    print(f"\nEntrenamiento terminado en {horas:.2f} h → {save_dir}")
    return save_dir, horas

SAVE_DIR, HORAS = entrenar(MODELO_BASE, NOMBRE_EXP)
BEST_PT = SAVE_DIR / "weights" / "best.pt"
json.dump({"save_dir": str(SAVE_DIR), "horas": HORAS}, open(RESULTADOS_DIR / "ultimo_entrenamiento.json", "w"))

### 6.1 Reanudar (solo si el entrenamiento se interrumpió)
Pon la ruta de la carpeta del experimento interrumpido y ejecuta.

In [ ]:
REANUDAR = False
CARPETA_INTERRUMPIDA = RUNS_DIR / NOMBRE_EXP   # ajusta si se llama ucsm_yolo26n2, etc.

if REANUDAR:
    modelo = YOLO(str(CARPETA_INTERRUMPIDA / "weights" / "last.pt"))
    modelo.train(resume=True)
    SAVE_DIR = Path(modelo.trainer.save_dir)
    BEST_PT = SAVE_DIR / "weights" / "best.pt"
    print("Reanudado y terminado:", SAVE_DIR)

## 7. Evaluación (validación y prueba)
Métricas para el Capítulo V: Precisión, Recall, F1, mAP@0.5 y mAP@0.5:0.95, global y por clase.

In [ ]:
import numpy as np

def evaluar(pesos, split, nombre):
    m = YOLO(str(pesos))
    met = m.val(data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=16, device=0,
                half=True, plots=True, project=str(RUNS_DIR), name=nombre, exist_ok=True, verbose=False)
    b = met.box
    p, r = float(b.mp), float(b.mr)
    global_ = {"split": split, "precision": p, "recall": r,
               "f1": 2 * p * r / (p + r) if (p + r) else 0.0,
               "mAP50": float(b.map50), "mAP50_95": float(b.map)}
    por_clase = []
    for i, c in enumerate(b.ap_class_index):
        pc, rc = float(b.p[i]), float(b.r[i])
        por_clase.append({"split": split, "clase": NOMBRES[int(c)], "precision": pc, "recall": rc,
                          "f1": 2 * pc * rc / (pc + rc) if (pc + rc) else 0.0,
                          "mAP50": float(b.ap50[i]), "mAP50_95": float(b.ap[i])})
    return global_, por_clase

globales, clases_met = [], []
for split in (["val", "test"] if TIENE_TEST else ["val"]):
    g, pc = evaluar(BEST_PT, split, f"{NOMBRE_EXP}_eval_{split}")
    globales.append(g); clases_met += pc

df_global = pd.DataFrame(globales).round(4)
df_clases = pd.DataFrame(clases_met).round(4)
display(df_global); display(df_clases)
df_global.to_csv(RESULTADOS_DIR / "metricas_globales.csv", index=False)
df_clases.to_csv(RESULTADOS_DIR / "metricas_por_clase.csv", index=False)

## 8. Velocidad de inferencia
Se mide preprocesamiento, inferencia y postprocesamiento por imagen (ms) y FPS, en **GPU FP16** y en **CPU**. Sirve para justificar el tiempo de inferencia en la tesis.

In [ ]:
from itertools import cycle, islice

IMGS_PRUEBA = sorted(p for p in (test_dir or val_dir).iterdir() if p.suffix.lower() in EXT_IMG)
assert IMGS_PRUEBA, "No hay imágenes para medir velocidad"

def medir_velocidad(pesos, device, n=150):
    m = YOLO(str(pesos))
    half = device != "cpu"
    for p in IMGS_PRUEBA[:10]:                              # calentamiento
        m.predict(str(p), imgsz=IMGSZ, device=device, half=half, verbose=False)
    tiempos = []
    for p in islice(cycle(IMGS_PRUEBA), n):
        r = m.predict(str(p), imgsz=IMGSZ, device=device, half=half, verbose=False)[0]
        tiempos.append(r.speed)
    df = pd.DataFrame(tiempos)
    total = df.sum(axis=1)
    return {"modelo": Path(pesos).parent.parent.name, "dispositivo": "GPU FP16" if half else "CPU",
            "pre_ms": df["preprocess"].mean(), "inferencia_ms": df["inference"].mean(),
            "post_ms": df["postprocess"].mean(), "total_ms": total.mean(),
            "p95_ms": total.quantile(0.95), "fps": 1000 / total.mean()}

velocidades = pd.DataFrame([medir_velocidad(BEST_PT, 0), medir_velocidad(BEST_PT, "cpu", n=50)]).round(2)
display(velocidades)
velocidades.to_csv(RESULTADOS_DIR / "velocidad_inferencia.csv", index=False)

## 9. (Opcional) Comparación YOLO26n vs YOLO26s
Con `ENTRENAR_COMPARACION = True` se entrena YOLO26s con la misma configuración. **Regla de decisión (eficiencia):** se elige `s` solo si mejora el mAP@0.5:0.95 en **≥ 0.02** y su latencia GPU sigue siendo **≤ 33 ms** (≥ 30 FPS); si no, se queda `n`.

In [ ]:
MODELO_ELEGIDO = BEST_PT

if ENTRENAR_COMPARACION:
    dir_s, horas_s = entrenar("yolo26s.pt", "ucsm_yolo26s")
    best_s = dir_s / "weights" / "best.pt"
    split_cmp = "test" if TIENE_TEST else "val"
    g_n, _ = evaluar(BEST_PT, split_cmp, "cmp_n")
    g_s, _ = evaluar(best_s, split_cmp, "cmp_s")
    v_n, v_s = medir_velocidad(BEST_PT, 0), medir_velocidad(best_s, 0)
    comparacion = pd.DataFrame([
        {"modelo": "YOLO26n", **g_n, "gpu_ms": v_n["total_ms"], "fps": v_n["fps"],
         "MB": BEST_PT.stat().st_size / 1e6, "horas": HORAS},
        {"modelo": "YOLO26s", **g_s, "gpu_ms": v_s["total_ms"], "fps": v_s["fps"],
         "MB": best_s.stat().st_size / 1e6, "horas": horas_s},
    ]).round(4)
    display(comparacion)
    comparacion.to_csv(RESULTADOS_DIR / "comparacion_n_vs_s.csv", index=False)
    if g_s["mAP50_95"] - g_n["mAP50_95"] >= 0.02 and v_s["total_ms"] <= 33:
        MODELO_ELEGIDO, NOMBRE_EXP = best_s, "ucsm_yolo26s"
print("Modelo elegido:", MODELO_ELEGIDO)

## 10. Exportación y copia al proyecto
- `yolo26_ucsm_best.pt` → para Ultralytics (lo usa el servidor Django).
- `.onnx` → portable (CPU/otras plataformas).
- `.engine` (TensorRT FP16) → máxima velocidad en la RTX 5060. Requiere `tensorrt`; si falla, se omite sin afectar lo demás. **El `.engine` solo funciona en la misma GPU y versión de TensorRT donde se creó.**

In [ ]:
FINAL_PT = MODELO_FINAL_DIR / "yolo26_ucsm_best.pt"
shutil.copy2(MODELO_ELEGIDO, FINAL_PT)
modelo_final = YOLO(str(FINAL_PT))

onnx_path = modelo_final.export(format="onnx", imgsz=IMGSZ, simplify=True)
print("ONNX:", onnx_path)

if EXPORTAR_TENSORRT:
    try:
        engine_path = modelo_final.export(format="engine", imgsz=IMGSZ, half=True, device=0, workspace=4)
        print("TensorRT:", engine_path)
    except Exception as e:
        print("TensorRT omitido:", e)

if DESTINO_DJANGO.exists():
    shutil.copy2(FINAL_PT, DESTINO_DJANGO / FINAL_PT.name)
    print("Copiado al proyecto Django:", DESTINO_DJANGO / FINAL_PT.name)
else:
    print("Copia manualmente", FINAL_PT, "a la carpeta modelos_ia del proyecto Django")

print("\nLíneas para el .env del servidor:")
print(f"YOLO_MODELO=modelos_ia/{FINAL_PT.name}")
print(f"YOLO_NOMBRE={NOMBRE_EXP.replace('ucsm_', '')}")
print("YOLO_CLASES=" + ",".join(NOMBRES))

## 11. Prueba visual y gráficas del entrenamiento

In [ ]:
import cv2

muestras = random.Random(SEMILLA).sample(IMGS_PRUEBA, min(6, len(IMGS_PRUEBA)))
fig, ejes = plt.subplots(2, 3, figsize=(16, 9))
for eje, p in zip(ejes.ravel(), muestras):
    r = modelo_final.predict(str(p), imgsz=IMGSZ, device=0, half=True, conf=0.5, verbose=False)[0]
    eje.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    eje.set_title(f"{p.name[:28]} · {len(r.boxes)} det · {sum(r.speed.values()):.1f} ms", fontsize=9)
    eje.axis("off")
for eje in ejes.ravel()[len(muestras):]:
    eje.axis("off")
plt.tight_layout(); plt.savefig(RESULTADOS_DIR / "predicciones_muestra.png", dpi=150); plt.show()

In [ ]:
from IPython.display import Image, display

dir_graficas = MODELO_ELEGIDO.parent.parent
for patron in ["results.png", "confusion_matrix_normalized.png", "*PR_curve.png", "*F1_curve.png"]:
    for archivo in sorted(dir_graficas.glob(patron)):
        print(archivo.name)
        display(Image(filename=str(archivo), width=900))
        shutil.copy2(archivo, RESULTADOS_DIR / archivo.name)

res = pd.read_csv(dir_graficas / "results.csv")
res.columns = [c.strip() for c in res.columns]
mejor = res.loc[res["metrics/mAP50-95(B)"].idxmax()]
print(f"Mejor época: {int(mejor['epoch'])} | mAP50-95 = {mejor['metrics/mAP50-95(B)']:.4f} | "
      f"mAP50 = {mejor['metrics/mAP50(B)']:.4f}")
print("\nTodo lo necesario para la tesis quedó en:", RESULTADOS_DIR)